[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ach0000/memorization_generalization_in_diffusion_models_tests/blob/main/tp/TP_debruiteur_empirique.ipynb)

# Le débruiteur optimal d'un jeu fini mémorise

On calcule exactement le débruiteur optimal $f^\star(y)=\mathbb E[x\mid y]$ lorsque la loi des images est la loi empirique d'un jeu d'entraînement de $N$ images : il ne fait que recopier ces images. Un réseau qui minimiserait parfaitement l'erreur d'entraînement ne généraliserait donc pas.

**Matériel** : tourne sur un ordinateur personnel (CPU) en quelques minutes ; sur Colab, exécuter d'abord la cellule d'installation (GPU facultatif).

In [ ]:
# Installation (Colab) : à exécuter en premier ; en local, ne fait que rendre le code importable
GITHUB_USER = "ach0000"  # propriétaire du fork
REPO = "memorization_generalization_in_diffusion_models_tests"
BRANCH = "main"
# seuls ces fichiers du dépôt sont téléchargés (motifs de git sparse-checkout)
FICHIERS = ['/code/',
            '/tp/tp_outils.py',
            '/tp/prepare_celeba80x80.py',
            '/denoisers/UNet/C_alpha4_beta4_line_80x80_blurredx2/']

import os, sys, subprocess
if "google.colab" in sys.modules:
    racine = os.path.join("/content", REPO)
    if not os.path.isdir(os.path.join(racine, ".git")):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                        "https://github.com/%s/%s.git" % (GITHUB_USER, REPO), racine], check=True)
    subprocess.run(["git", "-C", racine, "sparse-checkout", "set", "--no-cone", *FICHIERS], check=True)
    subprocess.run(["git", "-C", racine, "checkout", BRANCH], check=True)
    try:
        import pywt   # PyWavelets (générateur d'images C^alpha des auteurs) : absent de certaines images Colab
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyWavelets"], check=True)
else:
    racine = os.getcwd()
    while not os.path.exists(os.path.join(racine, "code", "main.py")) and os.path.dirname(racine) != racine:
        racine = os.path.dirname(racine)
os.makedirs(os.path.join(racine, "tp"), exist_ok=True)   # absent du clone partiel
os.chdir(os.path.join(racine, "tp"))
for d in ("code", "tp"):
    if os.path.join(racine, d) not in sys.path:
        sys.path.insert(0, os.path.join(racine, d))
print("Répertoire courant :", os.getcwd())

## 0. Réglages

`SOURCE = "calpha"` utilise les images synthétiques $C^\alpha$ des auteurs, produites en quelques secondes.
`SOURCE = "celeba"` utilise les visages CelebA (80×80, niveaux de gris), reconstruits au premier appel
(téléchargement d'environ 1,4 Go : à réserver à Colab). Toutes les graines sont fixées.

In [ ]:
%matplotlib inline
import numpy as np
import torch
import matplotlib.pyplot as plt
import tp_outils as T
T.nommer_figures("TP_debruiteur_empirique")   # figures nommées dans resultats/figures/

SOURCE = "calpha"   # ou "celeba"
GRAINE = 0
ALPHA = 4           # régularité des images C^alpha
K = 80              # côté des images (pixels)
print("calcul sur", T.DEVICE)

## 1. Les données

Un réservoir de 1 000 images d'entraînement (les jeux de taille $N$ en sont les $N$ premières) et 200 images
de test, distinctes. Les images sont dans $[0,1]$, de dimension $d = K^2 = 6\,400$.

In [ ]:
if SOURCE == "calpha":
    tout = T.images_calpha(ALPHA, 1200, K, graine=GRAINE)
else:
    tout = T.charger_celeba()[:1200]
X_reservoir, X_test = tout[:1000], tout[1000:]
d = X_reservoir[0].numel()
print("réservoir", tuple(X_reservoir.shape), " test", tuple(X_test.shape), " d =", d)
T.afficher_images(X_reservoir[:10], titre="images d'entraînement")

## 2. Le débruiteur optimal pour une loi empirique

Si $x$ suit la loi uniforme sur $\{x_1,\dots,x_N\}$ et $y = x + \sigma z$, $z\sim\mathcal N(0, I)$, la loi a
posteriori est portée par les $x_i$ :

$$w_i(y) = P(x = x_i \mid y) = \frac{\exp(-\|y - x_i\|^2 / 2\sigma^2)}{\sum_j \exp(-\|y - x_j\|^2 / 2\sigma^2)},
\qquad f^\star(y) = \mathbb E[x\mid y] = \sum_i w_i(y)\, x_i .$$

Les poids sont un *softmax* des distances (fonction `T.poids_empiriques`). On mesure leur concentration par le
**nombre effectif d'images** $n_{\text{eff}} = \exp\big(-\sum_i w_i \log w_i\big)$, qui vaut 1 si un seul
poids est non nul et $N$ si les poids sont égaux. On le calcule pour des images bruitées à partir d'images
d'entraînement et à partir d'images de test, pour $N = 1\,000$.

In [ ]:
def n_effectif(w):
    return torch.exp(-(w * torch.log(w.clamp_min(1e-30))).sum(dim=1))

X = X_reservoir[:1000]
sigmas = np.logspace(-2, 1.5, 15)
torch.manual_seed(GRAINE)
neff = {"entraînement": [], "test": []}
for sigma in sigmas:
    for nom, source in [("entraînement", X[:50]), ("test", X_test[:50])]:
        y = source + sigma * torch.randn_like(source)
        neff[nom].append(n_effectif(T.poids_empiriques(y, X, sigma)).mean().item())

plt.figure(figsize=(6, 4))
for nom, valeurs in neff.items():
    plt.loglog(sigmas, valeurs, "o-", label="y bruitée à partir d'une image de " + nom)
plt.axvline(1.0, color="k", ls=":", lw=0.8, label="bruit maximal de l'apprentissage (σ = 1)")
plt.xlabel("σ"); plt.ylabel("nombre effectif d'images"); plt.grid(alpha=0.3); plt.legend(fontsize=8)
plt.title("Concentration de la loi a posteriori (N = 1000)")
T.montrer("n_effectif")
for nom, valeurs in neff.items():
    print("n_eff (%s) :" % nom, np.round(valeurs, 1))

dist = torch.cdist(X.flatten(1), X.flatten(1)); dist.fill_diagonal_(float("inf"))
print("distance médiane d'une image à sa plus proche voisine : %.1f  (soit %.3f par pixel)"
      % (dist.min(dim=1).values.median(), dist.min(dim=1).values.median() / np.sqrt(d)))

## 3. Erreur de débruitage sur l'entraînement et sur le test

Comme dans la figure 1 de l'article : PSNR de sortie en fonction du PSNR d'entrée
$10\log_{10}(1/\sigma^2)$, pour des jeux de taille $N = 1, 10, 100, 1000$, sur des images d'entraînement (à
gauche) et de test (à droite). Le PSNR d'une restitution parfaite est infini : il est plafonné à 100 dB.

Si les poids des auteurs sont présents (dossier `denoisers/`), la courbe de test du réseau entraîné par les
auteurs sur $10^5$ images $C^4$ est ajoutée pour comparaison.

In [ ]:
import os
sig_range = np.logspace(0.3, 2.4, 12) / 255    # niveaux de bruit du notebook des auteurs
Ns = [1, 10, 100, 1000]
courbes_train, courbes_test = {}, {}
for N in Ns:
    XN = X_reservoir[:N].to(T.DEVICE)
    f = lambda y, s, XN=XN: T.debruiteur_empirique(y, XN, s)
    images_train = X_reservoir[:N].repeat(int(np.ceil(100 / N)), 1, 1, 1)[:100]
    courbes_train["N = %d" % N] = np.minimum(T.courbe_psnr(f, images_train, sig_range), 100)
    courbes_test["N = %d" % N] = T.courbe_psnr(f, X_test, sig_range)

chemin_reseau = "../denoisers/UNet/C_alpha4_beta4_line_80x80_blurredx2"
if SOURCE == "calpha" and ALPHA == 4 and os.path.isdir(chemin_reseau):
    reseau = T.charger_unet_auteurs("C_alpha4_beta4_line_80x80_blurredx2", 100000)
    f_reseau = lambda y, s: y - reseau(y)
    courbes_test["réseau des auteurs (N = 10^5)"] = T.courbe_psnr(f_reseau, X_test, sig_range)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
T.trace_psnr(courbes_train, sig_range, "Entraînement (plafonné à 100 dB)", ax=axes[0])
T.trace_psnr(courbes_test, sig_range, "Test", ax=axes[1])
T.montrer("psnr")
for nom, valeurs in courbes_test.items():
    print("%-30s test : %s" % (nom, np.round(valeurs, 1)))

## 4. Échantillonner avec le débruiteur optimal empirique

On utilise l'échantillonneur de l'article (algorithme 1, `T.echantillonner`, voir la section 3.2 de
l'énoncé). Il demande un débruiteur *aveugle*, qui ne reçoit pas $\sigma$ : `T.ResiduEmpiriqueAveugle` estime
$\sigma^2$ par $\min_i \|y - x_i\|^2 / d$. L'initialisation est, comme chez les auteurs, la moyenne des images
d'entraînement plus un bruit d'écart-type $\sigma_0$. Pour chaque échantillon, on cherche l'image
d'entraînement la plus proche (similarité cosinus, images centrées) et on compte combien d'images
d'entraînement distinctes ont été produites.

In [ ]:
def echantillons_empiriques(N, n=32, sigma0=1.0, graine=GRAINE):
    XN = X_reservoir[:N].to(T.DEVICE)
    init = T.init_echantillons(XN.mean(dim=0), n, sigma0=sigma0, graine=graine)
    ech, infos = T.echantillonner(T.ResiduEmpiriqueAveugle(XN), init, sigma_fin=0.01, graine=graine)
    sim, idx = T.plus_proches_voisins(ech, XN)
    return ech.cpu(), sim, idx, infos

for N in [10, 100]:
    for sigma0 in [1.0, 10.0]:
        ech, sim, idx, infos = echantillons_empiriques(N, sigma0=sigma0)
        print("N = %4d  sigma0 = %4.1f : similarité au plus proche voisin min %.4f, %2d images distinctes "
              "sur 32 échantillons (%d itérations)" % (N, sigma0, sim.min(), len(set(idx.tolist())),
                                                       infos["iterations"]))
ech, sim, idx, _ = echantillons_empiriques(100, n=10, sigma0=1.0)
T.afficher_images(torch.cat([ech, X_reservoir[idx]]), ncol=10,
                  titre="en haut : échantillons (N = 100, σ0 = 1) ; en bas : image d'entraînement la plus proche",
                  nom="echantillons")

## 5. Le jacobien du débruiteur optimal empirique

Pour la loi empirique, le jacobien vaut $\nabla f^\star(y) = \mathrm{Cov}[x \mid y] / \sigma^2 = \frac1{\sigma^2}
\sum_i w_i(y) (x_i - f^\star(y))(x_i - f^\star(y))^\top$ (`T.jacobien_empirique`), que l'on compare au
jacobien obtenu par différentiation automatique. On se place en un point $y$ situé à mi-chemin des deux images
d'entraînement les plus proches l'une de l'autre, là où la loi a posteriori n'est pas concentrée sur une
seule image.

In [ ]:
X10 = X_reservoir[:10]
sigma = 0.5
D10 = torch.cdist(X10.flatten(1), X10.flatten(1)); D10.fill_diagonal_(float("inf"))
i, j = divmod(D10.argmin().item(), 10)
print("paire la plus proche : images %d et %d, distance %.1f" % (i, j, D10[i, j]))
torch.manual_seed(GRAINE)
y = 0.5 * (X10[i:i+1] + X10[j:j+1]) + 0.01 * torch.randn(1, 1, K, K)
w = T.poids_empiriques(y, X10, sigma)[0]
print("poids a posteriori :", np.round(w.numpy(), 3))

J = T.jacobien_empirique(y, X10, sigma)
J_auto = T.jacobien(lambda u: T.debruiteur_empirique(u, X10.to(T.DEVICE), sigma), y, taille_lot=800).cpu()
print("écart maximal formule / différentiation automatique : %.2e  (max |J| = %.2e)"
      % ((J - J_auto).abs().max(), J.abs().max()))
valeurs, vecteurs, asym = T.decomposition_jacobien(J)
print("asymétrie relative : %.1e ; 5 premières valeurs propres :" % asym, np.round(valeurs[:5].numpy(), 4))
print("trace de J : %.4f" % valeurs.sum())
direction = (X10[i] - X10[j]).flatten(); direction /= direction.norm()
print("|cos| entre le premier vecteur propre et x_i - x_j : %.4f" % abs(vecteurs[:, 0] @ direction))
print("w_i w_j |x_i - x_j|² / σ² = %.2f" % (w[i] * w[j] * D10[i, j] ** 2 / sigma ** 2))
f_y = T.debruiteur_empirique(y, X10, sigma)
print("|f(y) - J y| / |f(y)| = %.3f" % ((f_y.flatten() - J @ y.flatten()).norm() / f_y.norm()))
T.afficher_vecteurs(torch.stack([vecteurs[:, 0], direction]), K, ["e_1(y)", "(x_i - x_j)/|x_i - x_j|"], ncol=2,
                    nom="vecteur_propre")

### Erreur optimale et trace du jacobien

L'erreur du débruiteur optimal vérifie $\mathrm{MSE}(f^\star, \sigma^2) = \mathbb E_y[\mathrm{tr}\,\mathrm{Cov}[x\mid y]]
= \sigma^2\, \mathbb E_y[\mathrm{tr}\,\nabla f^\star(y)]$. On compare, par Monte-Carlo sur 2 000 tirages
$x \sim$ loi empirique ($N = 100$), $y = x + \sigma z$, l'erreur quadratique mesurée et $\sigma^2\,\mathrm{tr}\,\nabla
f^\star(y)$, calculée sans former $J$ : $\mathrm{tr}\,\mathrm{Cov}[x\mid y] = \sum_i w_i \|x_i - f^\star(y)\|^2$.

In [ ]:
X100 = X_reservoir[:100].to(T.DEVICE)
torch.manual_seed(GRAINE)
lignes = []
for sigma in [0.5, 1.0, 2.0, 3.0, 5.0, 10.0]:
    x = X100[torch.randint(100, (2000,))]
    y = x + sigma * torch.randn_like(x)
    w = T.poids_empiriques(y, X100, sigma)
    f_y = (w @ X100.flatten(1))
    eqm = ((x.flatten(1) - f_y) ** 2).sum(dim=1).mean().item()
    trace_cov = (w * torch.cdist(f_y, X100.flatten(1)) ** 2).sum(dim=1).mean().item()
    lignes.append((sigma, eqm, trace_cov))
    print("σ = %5.1f   E|x - f(y)|² = %8.2f   σ² E[tr J] = %8.2f" % (sigma, eqm, trace_cov))